<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Attention_and_Prompted_probes_generalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformer_lens

# Setup files

Downloading necessary modules

In [ ]:
import transformer_lens
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


import numpy as np
import pandas as pd
import os
import json
import requests
from pathlib import Path
from typing import List, Dict
from typing import Optional, Literal
from collections import Counter
import random

import plotly.express as px
import matplotlib

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

## Downloading the Model

In [ ]:
model = transformer_lens.HookedTransformer.from_pretrained("Qwen/Qwen2.5-0.5B")

## Downloading the Train and Test datasets

In [ ]:
DATA_DIR = Path("data/high_stakes")
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_url = "https://pub-fd16e959a4f14ca48765b437c9425ba6.r2.dev/training/prompts_4x/train.jsonl"
train_path = DATA_DIR / "train.jsonl"

response = requests.get(train_url)
response.raise_for_status()

train_path.write_bytes(response.content)
print("Saved train data to", train_path)


MT_url = "https://pub-fd16e959a4f14ca48765b437c9425ba6.r2.dev/evals/dev/mt_balanced_apr_30.jsonl"
MT_dev_path = DATA_DIR / "MT_dev.jsonl"


response = requests.get(MT_url)
response.raise_for_status()
MT_dev_path.write_bytes(response.content)

print('Saved test data to', MT_dev_path)


In [ ]:
def load_jsonl(path) -> List[Dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

def label_to_int(x: str) -> int:
    if x == "high-stakes":
        return 1
    elif x == "low-stakes":
        return 0
    else:
        raise ValueError(f"Unexpected label: {x!r}")


def normalize_inputs(inputs_field: str) -> str:
    s = inputs_field.strip()


    if s.startswith('[') and '"role"' in s:
        try:
            messages = json.loads(s)
            parts = [f"{m['role']}: {m['content']}" for m in messages]
            return "\n".join(parts)
        except json.JSONDecodeError:

            return inputs_field
    else:

        return inputs_field


In [ ]:
train_rows = load_jsonl("data/high_stakes/train.jsonl")
dev_rows   = load_jsonl("data/high_stakes/MT_dev.jsonl")
len(train_rows), len(dev_rows)

In [ ]:
train_dataset = [{'text': normalize_inputs(row['inputs']), 'label': label_to_int(row['labels'])} for row in train_rows]
test_dataset = [{'text': normalize_inputs(row['inputs']), 'label': label_to_int(row['labels'])} for row in dev_rows]

train_texts = [train['text'] for train in train_dataset]
train_labels= [train['label'] for train in train_dataset]

test_texts = [test['text'] for test in test_dataset]
test_labels= [test['label'] for test in test_dataset]


combined = list(zip(train_texts, train_labels))

random.shuffle(combined)

train_texts, train_labels = zip(*combined)
train_texts = list(train_texts)
train_labels = list(train_labels)

In [ ]:
def create_dataloaders(
    activations: np.ndarray,
    labels: List[int],
    batch_size: int = 32,
    train_split: float = 0.8
):
    """Create train/val dataloaders from activations and labels"""

    # Convert to tensors
    X = torch.FloatTensor(activations)
    y = torch.FloatTensor(labels)

    # Create dataset
    dataset = TensorDataset(X, y)

    # Split train/val if needed
    if train_split < 1.0:
        train_size = int(train_split * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = torch.utils.data.random_split(
            dataset, [train_size, val_size]
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        return train_loader, val_loader
    else:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        return loader


In [ ]:
def get_activations(texts, model, layer_idx=-1, batch_size=8, pooling='last', pad_all=True):
    """
    Extract activations with different pooling strategies.

    Args:
        pad_all: If True and pooling='all', pad sequences to same length
    """
    model.eval()
    all_activations = []

    if layer_idx < 0:
      hook_name = f'blocks.{model.cfg.n_layers+layer_idx}.hook_resid_post'
    else:
      hook_name = f'blocks.{layer_idx}.hook_resid_post'

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        for text in batch_texts:
            captured = []

            def hook_fn(activation, hook):
                captured.append(activation.clone().cpu())

            with torch.no_grad():
                model.run_with_hooks(
                    text,
                    fwd_hooks=[(hook_name, hook_fn)]
                )

            hidden_states = captured[0][0]  # [seq_len, d_model]

            if pooling == 'last':
                act = hidden_states[-1, :]
            elif pooling == 'mean':
                act = hidden_states.mean(dim=0)
            elif pooling == 'first':
                act = hidden_states[0, :]
            elif pooling == 'all':
                act = hidden_states  # [seq_len, d_model]

            all_activations.append(act)
            del captured, hidden_states, act

        torch.cuda.empty_cache()

    # Concatenate based on pooling
    if pooling == 'all':
        if pad_all:
            # Pad to same length
            from torch.nn.utils.rnn import pad_sequence
            padded = pad_sequence(all_activations, batch_first=True)
            return padded.numpy()  # [num_texts, max_seq_len, d_model]
        else:
            return all_activations  # List of varying length tensors
    else:
        return torch.stack(all_activations, dim=0).numpy()

In [ ]:
class LinearProbe(nn.Module):


    def __init__(self, d_model: int, n_classes: int = 1):
        super().__init__()

        self.linear = nn.Linear(d_model, n_classes, bias = True)

    def forward(self, activations: torch.Tensor) -> torch.Tensor:
        """
        Args:
            activations: shape [batch, seq_len, d_model] OR [batch, d_model]
        Returns:
            logits: shape [batch, n_classes]
        """
        return self.linear(activations)



class AttentionProbe(nn.Module):
    """Attention probe: learns to weight token positions, then linear layer"""

    def __init__(self, d_model: int, n_classes: int = 2):
        super().__init__()
        # TODO: Initialize two components:
        # 1. Attention weights generator (activation -> single score per position)
        # 2. Classification layer (pooled activation -> logits)

        # Think: How do you go from [batch, seq, d_model] -> [batch, seq, 1] attention scores?
        # Then how do you use those to pool the activations?


    def forward(self, activations: torch.Tensor) -> torch.Tensor:
        """
        Args:
            activations: shape [batch, seq_len, d_model]
        Returns:
            logits: shape [batch, n_classes]
        """
        # TODO: Implement attention-based pooling + classification
        # Steps to think through:
        # 1. Generate attention scores for each position
        # 2. Softmax over sequence dimension
        # 3. Weighted sum of activations using attention weights
        # 4. Pass pooled activation through classifier
        pass

In [ ]:
class ProbeTrainer:
    def __init__(
        self,
        probe: nn.Module,
        learning_rate: float = 1e-3,
        weight_decay: float = 0.01,
        device: str = "cuda" if torch.cuda.is_available() else "cpu"
    ):
        self.probe = probe.to(device)
        self.device = device
        self.optimizer = torch.optim.Adam(probe.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.criterion = torch.nn.BCEWithLogitsLoss()

    def fit(
        self,
        train_activations: np.ndarray,
        train_labels: List[int],
        epochs: int = 100,
        batch_size: int = 32,
        patience: int = 10,
        train_split: float = 0.8  # Add this parameter for flexibility
    ):

        # Use create_dataloaders to handle train/val split
        train_loader, val_loader = create_dataloaders(
            train_activations,
            train_labels,
            batch_size=batch_size,
            train_split=train_split
        )

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):

            # Training
            self.probe.train()
            total_loss = 0
            num_batches = 0

            for x, y in train_loader:
                x = x.to(self.device)
                y = y.to(self.device).unsqueeze(1)

                self.optimizer.zero_grad()
                output = self.probe(x)
                loss = self.criterion(output, y)
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()
                num_batches += 1

            avg_train_loss = total_loss / num_batches

            # Validation
            self.probe.eval()
            val_loss = 0
            num_val_batches = 0

            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(self.device)
                    y = y.to(self.device).unsqueeze(1)
                    output = self.probe(x)
                    loss = self.criterion(output, y)
                    val_loss += loss.item()
                    num_val_batches += 1

            avg_val_loss = val_loss / num_val_batches

            # Early stopping
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

            if epoch % 10 == 0:
                print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    def evaluate(
        self,
        activations: np.ndarray,
        labels: List[int],
        batch_size: int = 32
    ) -> dict:
        """Evaluate probe on a dataset"""
        from sklearn.metrics import roc_auc_score

        loader = create_dataloaders(
            activations,
            labels,
            batch_size=batch_size,
            train_split=1.0
        )

        self.probe.eval()
        all_probs = []  # For AUROC - keep probabilities
        all_preds = []  # For binary metrics
        all_labels = []
        total_loss = 0

        with torch.no_grad():
            for x, y in loader:
                x = x.to(self.device)
                y = y.to(self.device)

                logits = self.probe(x)

                # Handle shape issues: ensure 1D for loss
                logits_squeezed = logits.squeeze()
                y_squeezed = y.squeeze()

                loss = self.criterion(logits_squeezed, y_squeezed)
                total_loss += loss.item()

                # Get probabilities (before thresholding) for AUROC
                probs = torch.sigmoid(logits_squeezed)

                # Binary predictions (threshold at 0.5)
                preds = (probs > 0.5).float()

                all_probs.append(probs.detach().cpu())
                all_preds.append(preds.detach().cpu())
                all_labels.append(y_squeezed.detach().cpu())

        # Concatenate all batches and convert to numpy with proper shapes
        probs = torch.cat(all_probs).numpy().flatten()
        preds = torch.cat(all_preds).numpy().flatten().astype(np.int32)
        labels_array = torch.cat(all_labels).numpy().flatten().astype(np.int32)

        # Ensure no NaN/Inf issues
        probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)

        # Compute metrics
        accuracy = accuracy_score(labels_array, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels_array, preds, average='binary', zero_division=0
        )
        auroc = roc_auc_score(labels_array, probs)

        return {
            'accuracy': float(accuracy),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1),
            'auroc': float(auroc),
            'loss': float(total_loss / len(loader))
        }

    def predict(
        self,
        activations: np.ndarray,
        batch_size: int = 32
    ) -> np.ndarray:
        """Get predictions for activations"""
        loader = create_dataloaders(
            activations,
            np.zeros(len(activations)),  # Dummy labels (not used)
            batch_size=batch_size,
            train_split=1.0
        )

        self.probe.eval()
        all_preds = []

        with torch.no_grad():
            for x, _ in loader:
                x = x.to(self.device)
                logits = self.probe(x)
                preds = (logits.sigmoid() > 0.5).float().squeeze()
                all_preds.append(preds.cpu().numpy())

        return np.concatenate(all_preds)

In [ ]:
train_activations = get_activations(train_texts, model, layer_idx=-1, batch_size=8, pooling='mean', pad_all=True)

In [ ]:
probe = LinearProbe(model.cfg.d_model)
probe_trainer = ProbeTrainer(probe)
probe_trainer.fit(train_activations, train_labels[:1000])

In [ ]:
test_activations = get_activations(test_texts, model, layer_idx=-1, batch_size=8, pooling='mean', pad_all=True)

In [ ]:
probe_trainer.evaluate(test_activations, test_labels)